1. Authentication and libs
2. Read Silver Delta Tables
3. Create Temporary Views
4. Gold Layer SQL Transformations
5. Gold Analytics
6. Save Gold Layer
7. Final Validation

1. Authentication and libs

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
%python
# Configuração de autenticação no ADLS Gen2
spark.conf.set(
    "fs.azure.account.key.nttcase.blob.core.windows.net",
    dbutils.secrets.get(scope="ntt-data", key="storage-key")
)

2. Read Silver Delta Tables

In [0]:
%python
locations_silver_df = spark.read \
    .format("delta") \
    .load(
        "wasbs://silver@nttcase.blob.core.windows.net/locations-silver/"
    )

In [0]:
%python
display(locations_silver_df)

In [0]:
%python
vaccinations_silver_df = spark.read \
    .format("delta") \
    .load(
        "wasbs://silver@nttcase.blob.core.windows.net/vaccinations-silver/"
    )

In [0]:
%python
display(vaccinations_silver_df)

3. Create Temporary Views

Locations Silver

In [0]:
locations_silver_df.createOrReplaceTempView(
    "vw_locations"
)

Vaccinations Silver

In [0]:
vaccinations_silver_df.createOrReplaceTempView(
    "vw_vaccinations"
)

QUESTÕES


QUESTION 1  What country(s) use more kind of vaccines

In [0]:
%sql

SELECT

    iso_code,

    location AS country,

    SIZE(SPLIT(vaccines, ',')) AS total_vaccine_types

FROM vw_locations

ORDER BY total_vaccine_types DESC

LIMIT 10;

QUESTION 2
Top 10 countries with the highest number of vaccinations

In [0]:
%sql

SELECT

    iso_code,

    country,

    YEAR(vaccination_date) AS vaccination_year,

    MONTH(vaccination_date) AS vaccination_month,

    SUM(total_vaccinations) AS total_vaccinations

FROM vw_vaccinations v

WHERE LOWER(v.country) NOT IN (

    'world',
    'asia',
    'africa',
    'europe',
    'north america',
    'south america',
    'oceania',
    'european union',
    'upper middle income',
    'lower middle income'

)

AND v.iso_code NOT IN (

    'OWID_UMC',
    'OWID_LMC',
    'OWID_HIC',
    'OWID_LIC'

)

GROUP BY

    iso_code,
    country,
    YEAR(vaccination_date),
    MONTH(vaccination_date)

ORDER BY total_vaccinations DESC

LIMIT 10;

QUESTION 3 Include in the top 10 country vaccinations per year, all the vaccine used during the fight 
against covid, ordering the top 10, first by most vaccine used and most vaccinated in the 
year.

In [0]:
%sql


SELECT

    v.iso_code,

    v.country,

    YEAR(v.vaccination_date) AS vaccination_year,

    l.vaccines,

    SIZE(SPLIT(l.vaccines, ',')) AS total_vaccine_types,

    SUM(v.total_vaccinations) AS total_vaccinations

FROM vw_vaccinations v

INNER JOIN vw_locations l
    ON v.iso_code = l.iso_code

WHERE LOWER(v.country) NOT IN (

    'world',
    'asia',
    'africa',
    'europe',
    'north america',
    'south america',
    'oceania',
    'european union',
    'upper middle income',
    'lower middle income'

)

AND v.iso_code NOT IN (

    'OWID_UMC',
    'OWID_LMC',
    'OWID_HIC',
    'OWID_LIC'

)

GROUP BY

    v.iso_code,
    v.country,
    YEAR(v.vaccination_date),
    l.vaccines

ORDER BY

    total_vaccine_types DESC,
    total_vaccinations DESC

LIMIT 10;